In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

sns.set(style="whitegrid", rc={"figure.figsize": (9,6)})
pd.options.display.float_format = '{:,.3f}'.format

# ============================================
# TEMPERATURE EFFECT OVER 24-HOUR CYCLE
# ============================================
data = pd.read_csv("hour.csv")

cols = ['dteday','season','yr','mnth','hr','holiday','weekday',
        'workingday','weathersit','temp','atemp','hum','windspeed',
        'casual','registered','cnt']
data = data[cols].copy()

# Sukuriam paros dalių kintamąjį
def to_day_period(h):
    if 0 <= h < 6: return 'night'
    if 6 <= h < 12: return 'morning'
    if 12 <= h < 18: return 'afternoon'
    return 'evening'
data['day_period'] = data['hr'].apply(to_day_period)

# Cikliniai terminai valandoms
data['hr_sin'] = np.sin(2*np.pi*data['hr']/24)
data['hr_cos'] = np.cos(2*np.pi*data['hr']/24)

# --- Vizualizacijos ---
plt.figure(figsize=(8,5))
sns.scatterplot(data=data, x='hr', y='cnt', alpha=0.35)
plt.title('Hour vs Total Rentals')
plt.xlabel('Hour'); plt.ylabel('Total rentals (cnt)')
plt.show()

plt.figure(figsize=(8,5))
sns.scatterplot(data=data, x='temp', y='cnt', alpha=0.35)
plt.title('Temperature vs Total Rentals')
plt.xlabel('Temperature (normalized)'); plt.ylabel('Total Rentals (cnt)')
plt.show()

g = sns.FacetGrid(data, col='day_period', col_wrap=2, height=4, sharex=True, sharey=True)
g.map_dataframe(sns.scatterplot, x='temp', y='cnt', alpha=0.35)
g.fig.suptitle('Temperature vs Total Rentals – per Day Periods', y=1.02)
for ax in g.axes.flatten():
    ax.set_xlabel('Temperature (normalized)')
    ax.set_ylabel('Total Rentals (cnt)')
plt.show()

# --- Modeliai ---
train, test = train_test_split(data, test_size=0.2, random_state=42)
y_train, y_test = train['cnt'], test['cnt']

models = {
    "M1": "cnt ~ hr",
    "M2": "cnt ~ hr + temp",
    "M3": "cnt ~ hr * temp",
    "M4": "cnt ~ hr_sin + hr_cos",
    "M5": "cnt ~ hr_sin + hr_cos + temp",
    "M6": "cnt ~ hr_sin + hr_cos * temp"
}

results = []
for name, formula in models.items():
    model = smf.ols(formula, data=train).fit()
    y_pred = model.predict(test)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    results.append({"Model": name, "R²": r2, "RMSE": rmse})

print("===== Baseline Models =====")
print(pd.DataFrame(results).to_string(index=False))

# --- Temperatūros poveikis per paros dalis ---
def fit_temp_model(subset_df, label):
    train, test = train_test_split(subset_df, test_size=0.2, random_state=42)
    formula = "cnt ~ temp + hr_sin + hr_cos"
    model = smf.ols(formula, data=train).fit()
    y_pred = model.predict(test)
    r2 = r2_score(test['cnt'], y_pred)
    rmse = np.sqrt(mean_squared_error(test['cnt'], y_pred))
    return {"Model": f"Temp × {label}", "R²": r2, "RMSE": rmse}

period_results = []
for period in ['morning', 'afternoon', 'evening', 'night']:
    subset = data[data['day_period'] == period]
    period_results.append(fit_temp_model(subset, period.capitalize()))

period_df = pd.DataFrame(period_results)
print("\n===== Temperature Effect by Day Period =====")
print(period_df.to_string(index=False))

plt.figure(figsize=(8,5))
sns.barplot(data=period_df, x='Model', y='R²', color='skyblue')
plt.title("Temperature Predictive Power (R²) by Day Period")
plt.xlabel("Day Period"); plt.ylabel("R²")
plt.show()

# ============================================
#  WIND SPEED EFFECT ON USER TYPES
# ============================================
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
sns.scatterplot(x='windspeed', y='casual', data=data, alpha=0.4)
sns.regplot(x='windspeed', y='casual', data=data, scatter=False, lowess=True, color='orange')
plt.title("Casual users vs windspeed")

plt.subplot(1,2,2)
sns.scatterplot(x='windspeed', y='registered', data=data, alpha=0.4)
sns.regplot(x='windspeed', y='registered', data=data, scatter=False, lowess=True, color='red')
plt.title("Registered users vs windspeed")
plt.tight_layout(); plt.show()

corr_total = data['windspeed'].corr(data['cnt'])
corr_casual = data['windspeed'].corr(data['casual'])
corr_registered = data['windspeed'].corr(data['registered'])
print(f"Koreliacija (windspeed, casual): {corr_casual:.4f}")
print(f"Koreliacija (windspeed, registered): {corr_registered:.4f}")
print(f"Koreliacija (windspeed, total cnt): {corr_total:.4f}")

# ============================================
#  WEATHER × WORKINGDAY INTERACTIONS
# ============================================
features = ["temp", "hum", "windspeed"]
for feature in features:
    X = pd.DataFrame({
        feature: data[feature],
        "workingday": data["workingday"],
        "interaction": data[feature] * data["workingday"]
    })
    y = data["casual"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = LinearRegression().fit(X_train, y_train)
    y_pred_test = model.predict(X_test)
    r2 = r2_score(y_test, y_pred_test)
    coef_inter = model.coef_[2]

    print(f"{feature} × workingday → R²={r2:.3f}, β_interaction={coef_inter:.2f}")

    # --- Vizualizacija ---
    plt.figure(figsize=(6,4))
    plt.scatter(X[feature], y, alpha=0.2, label="Casual rentals")
    for w, color in zip([0, 1], ['orange', 'blue']):
        x_vals = np.linspace(X[feature].min(), X[feature].max(), 100)
        X_plot = pd.DataFrame({
            feature: x_vals,
            "workingday": [w]*100,
            "interaction": x_vals * w
        })
        y_plot = model.predict(X_plot)
        label = "Weekend (0)" if w == 0 else "Working day (1)"
        plt.plot(x_vals, y_plot, color=color, label=label)
    plt.title(f"Casual rentals ~ {feature} × workingday")
    plt.xlabel(feature); plt.ylabel("Casual rentals")
    plt.legend(); plt.show()

# ============================================
#  CONCEPT DRIFT (2011 vs 2012)
# ============================================
data_2011 = data[data["yr"] == 0]
data_2012 = data[data["yr"] == 1]
features = ["temp", "atemp", "hum", "hr"]
target = "cnt"

def fit_model(df):
    X, y = df[features], df[target]
    model = LinearRegression().fit(X, y)
    y_pred = model.predict(X)
    r2 = r2_score(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    return model, r2, rmse

m2011, r2_2011, rmse_2011 = fit_model(data_2011)
m2012, r2_2012, rmse_2012 = fit_model(data_2012)

print(f"2011 model: R²={r2_2011:.3f}, RMSE={rmse_2011:.2f}")
print(f"2012 model: R²={r2_2012:.3f}, RMSE={rmse_2012:.2f}")

# --- Cross-year testing ---
y_pred_2011_on_2012 = m2011.predict(data_2012[features])
y_pred_2012_on_2011 = m2012.predict(data_2011[features])
print(f"2011→2012 R²: {r2_score(data_2012[target], y_pred_2011_on_2012):.3f}")
print(f"2012→2011 R²: {r2_score(data_2011[target], y_pred_2012_on_2011):.3f}")

# --- Scatter plots ---
plt.figure(figsize=(12,10))
plt.subplot(2,1,1)
plt.scatter(data_2012["cnt"], y_pred_2011_on_2012, alpha=0.4, color='orange')
plt.plot([0, 1000], [0, 1000], 'k--')
plt.title("2011 model tested on 2012")

plt.subplot(2,1,2)
plt.scatter(data_2011["cnt"], y_pred_2012_on_2011, alpha=0.4, color='blue')
plt.plot([0, 1000], [0, 1000], 'k--')
plt.title("2012 model tested on 2011")
plt.tight_layout(); plt.show()

# --- Boxplot ---
data["day_period"] = pd.cut(data["hr"], bins=[0,6,12,18,24],
                            labels=["night","morning","day","evening"], right=False)
plt.figure(figsize=(8,5))
sns.boxplot(x="day_period", y="cnt", hue="yr", data=data, palette=["blue","orange"])
plt.title("Rental distribution by time of day (2011 vs 2012)")
plt.xlabel("Day period"); plt.ylabel("Total rentals (cnt)")
plt.legend(title="Year", labels=["2011","2012"])
plt.show()

# ============================================
# CASUAL / TOTAL RATIO ANALYSIS
# ============================================
data["casual_ratio"] = data["casual"] / data["cnt"]
data = data.replace([np.inf, -np.inf], np.nan).dropna(subset=["casual_ratio"])
features = ["temp", "hum", "windspeed", "hr"]

X, y = data[features], data["casual_ratio"]
model = LinearRegression().fit(X, y)
y_pred = model.predict(X)

r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))
print(f"R²={r2:.3f}, RMSE={rmse:.3f}")

# --- Vizualizacijos ---
plt.figure(figsize=(6,5))
plt.scatter(y, y_pred, alpha=0.4, color='orange')
plt.plot([0, 1], [0, 1], '--', color='black')
plt.title("Actual vs Predicted Casual Ratio")
plt.xlabel("Actual"); plt.ylabel("Predicted")
plt.show()

plt.figure(figsize=(10,5))
sns.lineplot(x="hr", y="casual_ratio", data=data, color='orange', linewidth=2)
plt.title("Average Casual Ratio by Hour of Day")
plt.xlabel("Hour"); plt.ylabel("Casual / Total Rentals")
plt.show()

plt.figure(figsize=(7,5))
sns.boxplot(x="weathersit", y="casual_ratio", data=data, palette="coolwarm")
plt.title("Casual Ratio by Weather Condition")
plt.xlabel("Weather Situation (1=Good → 4=Bad)")
plt.ylabel("Casual / Total Rentals")
plt.show()